# P17 · Primera investigación cuantitativa integral
**QuantLab · Módulo 01 · Fundamentos financieros**

## Objetivo
Investigar si los movimientos recientes de BTC/EUR en Kraken anticipan los de OKX, comprobar si una regla long-only sobre OKX resulta económicamente viable y documentar un resultado reproducible. Bitvavo se usa como contraste descriptivo. Este cuaderno es un tutorial, no una estrategia para operar.

**Modo por defecto:** datos sintéticos, con un desfase deliberadamente introducido. Para una investigación empírica, importar una captura real de P06. No confundir ambos modos; el CSV puede contener simulaciones también.

**Resultado de referencia (semilla 1701):** 21.453 cotizaciones artificiales; R² fuera de muestra 32,94%; 143 operaciones completas en validación, +9,95 EUR brutos y −183,55 EUR netos condicionados. 15 señales con profundidad insuficiente y 14 ventanas de validación sin datos completos. No se aprueba la hipótesis económica. El resultado no es evidencia real.

**Con datos reales (E2, captura de 30 minutos del 20-11-2026):** Kraken no anticipa a OKX; no hay predictibilidad ni viabilidad. Los ejercicios están resueltos en la sección 11.

Las funciones están en **`src/quantlab/p17/integrated_research.py`** y sus pruebas en **`tests/test_integrated_research.py`** (`pytest` desde la raíz del proyecto).

## 1. Protocolo antes de mirar los resultados
- H0: la señal de Kraken no tiene relación lineal positiva con el retorno futuro de OKX. H1: relación positiva.
- Hipótesis económica: media del PnL neto positiva, con intervalo de incertidumbre por bloques por encima de cero. Si no se cumple, no se aprueba para operar.
- Señal fijada: retorno del mid de Kraken en tres segundos. Entrada un segundo después de decidir; salida cuatro segundos después de entrar. Umbral: percentil 70 de la señal en desarrollo, exigir además señal positiva.
- Rejilla de decisiones cada cinco segundos; posiciones no solapadas. Corte 60% desarrollo, 40% reservado, descartando operaciones que cruzan el corte. No elegir el mejor lag mirando validación.
- Comprar al ask y vender al bid, cantidad fija, exigir profundidad L1, descontar comisión por lado y reserva adversa. Sin cortos, sin apalancamiento, sin órdenes reales. No modelamos las colas ni garantizamos ejecución.
- Contraste principal: regresión HAC (errores robustos a heterocedasticidad y autocorrelación, aproximación asintótica), e intervalo bootstrap por bloques del PnL por oportunidad. No hay anualización de una captura corta. El resto de análisis es exploratorio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display
from pathlib import Path
import hashlib, json, inspect, sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p17.integrated_research import (audit_capture, block_ci, circular_shift_null, decision, evaluate,
                                              fee_sensitivity, fit_hac, lead_lag, opportunities, opportunity_metrics,
                                              partial_fill_engine, predictability, register_protocol, returns_bps,
                                              run_study, session_lead_lag, summarize_pnl, synchronize,
                                              synthetic_quotes, validate)
from quantlab.p12.factors import manual_alpha_beta

DATA_MODE = 'synthetic'  # 'csv' para captura P06
CSV_PATH = 'p06_quotes.csv'
CSV_PROVENANCE = ''  # obligatorio en csv: origen, fecha, modo original y condiciones de captura
SYMBOL = 'BTC/EUR'
EXCHANGES = ['kraken','okx','bitvavo']
SEED = 1701
QTY = 0.01  # BTC
FEE_BPS = 10.0  # por lado, hipotético
EXTRA_BPS = 1.0  # reserva adversa por lado
MAX_AGE_S = 1.5
LOOKBACK = 3
DELAY = 1
HOLD = 4
STRIDE = DELAY + HOLD
INITIAL_EQUITY = 1000.0
EXPORT = False
plt.rcParams.update({'figure.figsize':(9,3.5),'axes.grid':True,'grid.alpha':.2,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})

## 2. Datos y procedencia
La captura usa `received_at` (momento de recepción), `exchange`, `symbol`, `bid`, `ask`, `bid_size`, `ask_size`. El tiempo de recepción es el que determina qué conocía el sistema. Los precios se expresan en EUR/BTC; las cantidades en BTC. No se mezclan distintos pares ni se introducen datos futuros.

Generador: camino común con ruido, retrasos de 0, 3 y 2 segundos, spreads distintos y pérdidas ocasionales de mensajes. No replica el comportamiento real de los exchanges. En modo CSV reutilizar la captura de P06, con `EXPORT=True` en aquel notebook, conservando su origen y timestamps. Una captura breve puede no ser suficiente; el análisis se detiene si faltan oportunidades.

In [ ]:
print(inspect.getsource(synthetic_quotes))

if DATA_MODE=='synthetic':
    raw=synthetic_quotes(seed=SEED); provenance={'mode':DATA_MODE,'seed':SEED,'source':'synthetic_quotes()'}
elif DATA_MODE=='csv':
    if not CSV_PROVENANCE.strip(): raise ValueError('Documenta el origen y modo original del CSV.')
    raw=pd.read_csv(CSV_PATH)
    provenance={'mode':DATA_MODE,'source':CSV_PATH,'description':CSV_PROVENANCE,'sha256':hashlib.sha256(Path(CSV_PATH).read_bytes()).hexdigest()}
else: raise ValueError('Modo desconocido')
print(provenance)
print('Filas originales:',len(raw))
display(raw.head())

## 3. Calidad y sincronización causal
Rechazamos duplicados de tiempo por exchange, precios cruzados, tamaños no positivos y entradas inválidas. Creamos una rejilla UTC de un segundo usando únicamente la última cotización recibida **antes o en** el instante de decisión. Si es demasiado antigua, queda ausente; no usamos interpolación ni relleno ilimitado. El orden observado puede reflejar diferencias de latencia de captura.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿la captura es válida y la rejilla causal?</b>

`validate` se detiene con un error si faltan columnas, falta algún exchange, hay tiempos inválidos o duplicados por exchange, precios o tamaños no finitos o ≤ 0, o libros cruzados.

`synchronize` solo usa la última cotización recibida **antes o en** cada segundo y la descarta si tiene más de 1,5 s; se detiene si el tramo común tiene menos de 500 segundos.

</div>

In [ ]:
print(inspect.getsource(validate))
print(inspect.getsource(synchronize))

quotes=validate(raw, SYMBOL, EXCHANGES); g=synchronize(quotes, EXCHANGES, MAX_AGE_S)
cut=g.index[int(len(g)*.60)]
print('Corte:',cut,'| segundos:',len(g))
display(pd.DataFrame({'coverage_pct':[g[e+'_mid'].notna().mean()*100 for e in EXCHANGES], 'spread_bps':[g[e+'_spread_bps'].mean() for e in EXCHANGES]},index=EXCHANGES).round(3))

## 4. Price Discovery: exploración solo en desarrollo
Calculamos retornos simples de un segundo en puntos básicos. Correlación entre retorno de Kraken en t y retorno de otro exchange en t+k: lag positivo indica que Kraken precede estadísticamente al otro. No demuestra causalidad ni identifica por sí solo el mercado que descubre el precio. Evitamos que el retorno desplazado cruce el corte.

In [ ]:
r=returns_bps(g, EXCHANGES)
dev_r=r.loc[r.index<cut]
lead=lead_lag(dev_r, 'kraken', ['okx','bitvavo'], 5)
ax=lead.plot(marker='o'); ax.axvline(0,color='grey',ls='--'); ax.set(xlabel='Lag k (segundos)',ylabel='Correlación Pearson',title='Lead-lag de retornos · desarrollo'); plt.tight_layout(); plt.show()
print('Lag de mayor correlación (descriptivo):',lead.idxmax().to_dict())

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Simulación: el desfase programado se recupera</b>

- 7.199 segundos sincronizados, con cobertura del 99,2–99,4 % (se pierde ~0,8 % de mensajes) y el corte a la 01:12 UTC.
- La correlación máxima está en **k = 3 para OKX y k = 2 para Bitvavo**: exactamente los retrasos que programa el generador. Con un retraso así, la señal de Kraken tiene que predecir: es el caso favorable.

</div>

## 5. Definir observaciones, señal y objetivo
En cada decisión t la señal utiliza precios de t y t−3. El objetivo bruto es el retorno del mid de OKX de t+1 a t+5. Las cotizaciones futuras solo se emplean como resultado de la simulación. Rejilla cada 5 s; no se solapan ventanas de mantenimiento. El umbral se aprende exclusivamente en desarrollo. Todos los períodos sin datos necesarios se descartan y cuantifican; no se asumen retornos cero por falta de datos.

In [ ]:
print(inspect.getsource(opportunities))

all_op=opportunities(g, cut, LOOKBACK, DELAY, HOLD, QTY)
display(all_op.groupby('split')['observed'].agg(['count','sum','mean']).round(3))
d=all_op.loc[all_op.observed].copy()
dev=d.loc[d.split.eq('dev')].copy(); val=d.loc[d.split.eq('val')].copy()
if min(len(dev),len(val))<60: raise ValueError('Captura insuficiente: se requieren 60 oportunidades observadas por segmento como mínimo didáctico.')
threshold=max(0.,dev.signal_bps.quantile(.7))
print('Umbral de desarrollo (bps):',round(threshold,4),'| N desarrollo / validación',len(dev),len(val))

## 6. Evaluar predictibilidad
Regresión lineal de retorno futuro sobre señal con errores HAC de hasta 5 observaciones. Se informa coeficiente, IC (correlación), p-valor bilateral y R². El criterio fijado exige coeficiente positivo y p bilateral < 0,05 en validación; es conservador frente al contraste unilateral. El R² fuera de muestra usa coeficientes congelados de desarrollo y el predictor constante de media de desarrollo como referencia. No se selecciona el modelo usando los datos reservados.

In [ ]:
print(inspect.getsource(predictability))

stats, oos, a, b = predictability(dev, val)
display(stats.round(5)); print('R2 OOS vs media desarrollo:',round(oos,5))
ax=val.plot.scatter(x='signal_bps',y='target_bps',alpha=.35,s=12); xx=np.linspace(val.signal_bps.min(),val.signal_bps.max(),100); ax.plot(xx,a.params['const']+a.params['signal_bps']*xx,color='tab:orange',label='Modelo congelado'); ax.set(xlabel='Señal Kraken (bps)',ylabel='Retorno futuro OKX (bps)',title='Predictibilidad · validación'); ax.legend(); plt.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La señal predice: R² fuera de muestra del 32,9 %</b>

| Muestra | n | IC | Pendiente | p | R² |
|---|---|---|---|---|---|
| Desarrollo | 843 | 0,591 | 0,698 | ≈ 0 | 0,349 |
| Validación | 561 | 0,574 | 0,648 | ≈ 0 | 0,329 |

- De 862 y 575 ventanas, se observan completas **843 y 561**: 19 y 14 se descartan por falta de datos, sin rellenarlas.
- El umbral de desarrollo (percentil 70) es **0,85 bps**.
- Con el modelo congelado de desarrollo, el R² fuera de muestra es **32,9 %**. La hipótesis estadística H1 se cumple, como era de esperar con un retraso programado de 3 s.

</div>

## 7. Motor de ejecución y costes
Cantidad fija. Se compra al ask ajustado y vende al bid ajustado. Comisión sobre cada nominal ejecutado. PnL bruto de referencia = q × (mid salida − mid entrada). PnL neto = q × (bid ajustado − ask ajustado) − comisiones. Así no contamos el spread dos veces.

La señal determina intención, no el resultado. Si falta profundidad en cualquiera de los dos extremos, no contamos esa operación como ejecutada y marcamos su PnL como desconocido: **excluir por profundidad futura es una selección ex post**, no un filtro operativo. Por eso presentamos una simulación condicionada y una tasa de cobertura, y no aprobamos la viabilidad si hay fallos de profundidad en señales seleccionadas. Una simulación real necesitará resolver las salidas no ejecutadas y la latencia. Las oportunidades observadas sin señal sí tienen PnL cero. El benchmark es long en cada oportunidad con las mismas restricciones y costes. Las curvas son capital contable hipotético (cantidad fija), no una simulación autofinanciada: no se impone restricción de efectivo ni se modela inventario tras salidas fallidas. Un benchmark puede quedar por debajo del nominal requerido y no sería ejecutable con ese capital.

In [ ]:
print(inspect.getsource(evaluate))
print(inspect.getsource(summarize_pnl))

pv=evaluate(val, threshold, QTY, FEE_BPS, EXTRA_BPS); pdv=evaluate(dev, threshold, QTY, FEE_BPS, EXTRA_BPS)
bench=evaluate(val, threshold, QTY, FEE_BPS, EXTRA_BPS, always=True)
summary=pd.DataFrame({'development':summarize_pnl(pdv, INITIAL_EQUITY),'validation':summarize_pnl(pv, INITIAL_EQUITY),
                      'always_long_validation':summarize_pnl(bench, INITIAL_EQUITY)})
display(summary.round(5))
ax=(INITIAL_EQUITY+pd.DataFrame({'Señal bruto':pv.gross_pnl.dropna().cumsum(),'Señal neto':pv.net_pnl.dropna().cumsum(),'Siempre long neto':bench.net_pnl.dropna().cumsum()})).plot(); ax.set(title='Capital contable hipotético · validación',ylabel='EUR, cantidad fija',xlabel='UTC'); plt.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Predecir no basta: todas las operaciones pierden</b>

| Validación | Señal | Siempre long |
|---|---|---|
| Señales / ejecuciones / fallos de profundidad | 158 / **143** / **15** | 561 / 515 / 46 |
| PnL bruto | **+9,95 €** | −2,21 € |
| PnL neto condicionado | **−183,55 €** | −699,10 € |
| Neto medio por ejecución | **−21,34 bps** | −22,57 bps |
| Operaciones ganadoras | **0 %** | 0 % |

- La señal acierta la dirección (bruto positivo, mucho mejor que siempre long), pero el movimiento bruto medio por operación es de **~1,2 bps** (9,95 € / 143 / 6 € de nominal) y la ida y vuelta cuesta unos 22 bps: 2 × 10 bps de comisión + 2 × 1 bp de reserva + el spread.
- Es el mismo umbral que vimos en P08: con 10 bps por lado, el mid tiene que subir unos 21,7 bps solo para empatar.
- 15 señales (9,5 %) no tienen profundidad en el mejor nivel: su resultado es desconocido, y por eso el PnL es **condicionado**.

</div>

## 8. Incertidumbre y sensibilidad
Bootstrap de bloques circulares de 10 oportunidades contiguas en la tabla observada, 2.000 réplicas, intervalo percentil 95%. Los huecos de captura impiden interpretar esos bloques como intervalos de tiempo continuos; se trata de un diagnóstico aproximado. Si faltan PnL por profundidad, el intervalo se calcula solo sobre resultados disponibles y no permite aprobar viabilidad. No estamos estimando una probabilidad real de éxito. Sensibilidad a comisiones: el umbral y toda la regla se mantienen fijos.

In [ ]:
print(inspect.getsource(block_ci))

ci=block_ci(pv.net_pnl, seed=SEED+1)
sens=fee_sensitivity(val, threshold, (0,1,2,5,10,20), qty=QTY, extra_bps=EXTRA_BPS)
print('IC 95% PnL medio observado (EUR/oportunidad):',ci.round(6)); display(sens.round(3))
ax=sens.plot(marker='o'); ax.axhline(0,color='grey',ls='--'); ax.set(xlabel='Comisión por lado (bps)',ylabel='PnL acumulado (EUR)',title='Sensibilidad a comisiones · validación'); plt.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Ni sin comisiones sale positivo</b>

- **IC 95 % del PnL medio por oportunidad: [−0,373; −0,299] €**. Todo el intervalo está por debajo de 0.
- **Sensibilidad a comisiones:** con 0 bps por lado, la regla todavía pierde **−11,54 €**: el spread y la reserva adversa (1 bp por lado) ya superan lo que anticipa la señal. Con 20 bps, −355,56 €.

</div>

## 9. Atribución: ¿exposición direccional o ventaja?
Regresamos el PnL neto por oportunidad (normalizado por el nominal de entrada) sobre el retorno futuro del mid de OKX. El intercepto es un alpha relativo al benchmark de esas ventanas, no un Alpha CAPM ni una atribución de cartera completa. Las oportunidades sin posición aportan cero; las fallidas se excluyen. No se anualiza.

In [ ]:
print(inspect.getsource(decision))

attr=pv.loc[pv.net_bps.notna()].copy()
model=fit_hac(attr, 'target_bps', 'net_bps')
display(pd.DataFrame({'estimate':model.params,'p_value':model.pvalues}).round(5))
filled=pv.loc[pv.filled]
print('Solo ejecuciones (alpha, beta) frente al retorno del mid de OKX:', [round(v, 4) for v in manual_alpha_beta(filled.net_bps, filled.target_bps)])
verdict=decision(b, pv, all_op, ci)
stat_ok, econ_ok = verdict['stat_ok'], verdict['econ_ok']
print('Evidencia estadística según protocolo:',stat_ok)
print('Criterio económico conservador:',econ_ok)
print('DECISIÓN:', verdict['decision'])

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La pendiente negativa es selección, no exposición</b>

- Con todas las oportunidades, el intercepto es **−5,79 bps** y la pendiente, **−2,30**. Una pendiente negativa parece absurda (¿la estrategia gana cuando el mercado baja?), pero es un efecto de **selección**:
  - Las ventanas con mayor retorno futuro son las que activan la señal, y esas pagan unos 22 bps de coste.
  - Las demás, con menos retorno, valen 0.
  - Así, más retorno de mercado se asocia a más PnL negativo.
- **Solo con las ejecuciones**, la beta es **1,00** y el alpha, **−22,5 bps**: cada operación sigue al mercado uno a uno y paga el coste completo.
- **Decisión:** evidencia estadística sí; criterio económico no (fallos de profundidad, ventanas sin datos e IC por debajo de 0). **No se aprueba la hipótesis económica.**

</div>

## 10. Comprobaciones del motor
Estas pruebas verifican identidades matemáticas y el corte temporal, no prueban que los datos representen un mercado real.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: rejilla sin futuro, corte, no solapamiento, reconciliación y monotonía en costes
f=pv.loc[pv.filled]
checks={
 'timestamps sin futuro':all((g[e+'_age_s'].dropna()>=0).all() for e in EXCHANGES),
 'antigüedad limitada':all((g[e+'_age_s'].dropna()<=MAX_AGE_S).all() for e in EXCHANGES),
 'corte desarrollo':bool((dev.exit_time<cut).all()),
 'corte validación':bool((val.index>=cut).all()),
 'sin posiciones solapadas':bool((val.index.to_series().diff().dropna()>=pd.Timedelta(seconds=STRIDE)).all()),
 'umbral solo desarrollo':np.isclose(threshold,max(0.,dev.signal_bps.quantile(.7))),
 'reconciliación PnL':np.allclose(f.net_pnl,QTY*(f.sell_px-f.buy_px)-f.fees),
 'costes no negativos':bool((f[['spread_slippage','fees']]>=0).all().all()),
 'sin señal = cero':bool(pv.loc[~pv.intent,'net_pnl'].eq(0).all()),
 'salida no ejecutada no imputada':bool(pv.loc[pv.failed_depth,'net_pnl'].isna().all()),
 'comisión creciente no mejora PnL':bool((sens.diff().dropna()<=1e-10).all()),
 'OBI acotado':bool(g.kraken_obi.dropna().between(-1,1).all())}
assert all(checks.values()),checks
print(f'{sum(checks.values())}/{len(checks)} comprobaciones superadas')
if EXPORT:
    folder=Path('quantlab_exports')/f'p17_{DATA_MODE}'; folder.mkdir(parents=True,exist_ok=True)
    pv.to_csv(folder/'validation_opportunities.csv'); stats.to_csv(folder/'predictability.csv'); summary.to_csv(folder/'performance.csv')
    (folder/'manifest.json').write_text(json.dumps({**provenance,'cut':cut.isoformat(),'threshold_bps':float(threshold),'fee_bps':FEE_BPS,'extra_bps':EXTRA_BPS,'qty':QTY,'lookback':LOOKBACK,'delay':DELAY,'hold':HOLD,'max_age_s':MAX_AGE_S,'decision_stat':stat_ok,'decision_econ':econ_ok},indent=2),encoding='utf-8')

## 11. Trabajo autónomo
1. E1 — Documenta una hipótesis económica alternativa y el mecanismo que podría producirla. Escribe el protocolo **antes** de analizar el test.
2. E2 — Captura datos reales con P06 durante suficientes sesiones. Conserva archivo original, metadata, UTC y errores de captura; indica qué representa una observación.
3. E3 — Programa una auditoría por exchange: duplicados, huecos, spreads, timestamps, edad y profundidad. Comprueba que ninguna rejilla incorpora datos futuros.
4. E4 — Compara el lead-lag en dos sesiones de desarrollo. Propón una señal OBI alternativa sin seleccionar parámetros con validación.
5. E5 — Implementa un null por desplazamientos circulares de bloques dentro de desarrollo; explica por qué barajar observaciones individualmente puede destruir dependencia temporal. Reserva una nueva muestra para cualquier señal nueva.
6. E6 — Implementa un modelo de salida parcial y de inventario para fallos de profundidad. Aplica comisiones reales verificadas y una reserva adversa; compara con P08.
7. E7 — Recalcula métricas P09 y atribución P12 con un benchmark pertinente. No anualices una captura de minutos u horas. Incluye la fracción de datos descartados.
8. E8 — Elabora un informe de una página: hipótesis, fuente, unidades, periodo, método, evidencia, costes, límites, decisión y próximos experimentos. Un resultado negativo reproducible es una investigación válida.

### E1 · Hipótesis alternativa registrada antes del análisis
**Hipótesis:** cuando el mid de OKX queda por debajo del de Kraken más de lo habitual, el de OKX **converge** hacia el de Kraken en los segundos siguientes.

**Mecanismo:** OKX tiene menos liquidez en BTC/EUR y su libro se actualiza peor. Los arbitrajistas que operan en los dos mercados cierran la diferencia comprando en el barato, así que OKX sube.

El protocolo se fija en un diccionario y se registra su huella SHA-256. Cualquier cambio posterior cambia la huella, y queda claro que no se ajustó después de ver los datos. Se evaluará en una **captura nueva**, no en la del E2.

In [ ]:
print(inspect.getsource(register_protocol))

protocol_e1 = {
    'hipotesis': 'Si (mid Kraken − mid OKX) / mid OKX supera su percentil 70 de desarrollo, el mid de OKX sube de t+1 a t+5',
    'h0': 'La diferencia Kraken − OKX no tiene relación lineal positiva con el retorno futuro del mid de OKX',
    'mecanismo': 'Actualización más lenta y menor liquidez en OKX; los arbitrajistas cierran la diferencia',
    'senal': '(kraken_mid − okx_mid) / okx_mid · 1e4 en t (bps)', 'umbral': 'percentil 70 en desarrollo y > 0',
    'ejecucion': 'compra al ask de OKX en t+1, venta al bid en t+5, 0,01 BTC, profundidad L1 obligatoria',
    'costes': '10 bps por lado + 1 bp de reserva; sensibilidad a 20 bps', 'muestra': 'captura nueva de al menos 30 minutos, 60 % / 40 %',
    'criterio_estadistico': 'pendiente > 0 y p HAC < 0,05 en validación',
    'criterio_economico': 'IC bootstrap por bloques del PnL medio > 0, sin fallos de profundidad ni ventanas sin datos',
}
print('Huella del protocolo E1:', register_protocol(protocol_e1))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Protocolo fijado</b>

- Hipótesis, H0, mecanismo, señal, umbral, ejecución, costes, muestra y los dos criterios de decisión quedan escritos **antes** de mirar ningún resultado de esa señal. Su huella SHA-256 queda impresa arriba.
- **Por qué una captura nueva:** la del E2 ya se ha mirado para la hipótesis principal. Usarla para elegir una señal nueva y además validarla convertiría la validación en desarrollo.
- **Riesgo conocido:** si OKX está más bajo porque su libro es más ancho (spread medio de 2,2 bps frente a 0,18 en Kraken, E3), la "diferencia" puede ser solo el spread, y comprar al ask lo anula.

</div>

### E2 · Captura real con P06
Cambia `E2_MODE` a `'capture'` para capturar Kraken, OKX y Bitvavo con `collect_live` de P06 (una ronda por segundo, peticiones simultáneas, unos 30 minutos con 1.800 rondas). La captura se guarda en `data/raw/`, que **no se versiona** (`.gitignore`). Con `'csv'` se reutiliza un fichero ya capturado. Por defecto no se consulta ningún endpoint.

**Qué representa una observación:** la mejor oferta y demanda (L1) de un exchange, tal como la **recibimos** en `received_at`, con `request_at` (antes de pedirla) y `exchange_at` (la hora del evento según el exchange, solo OKX).

In [ ]:
E2_MODE = None  # None | 'capture' | 'csv'
E2_ROUNDS = 1800
E2_FILE = 'p17_quotes_20261120.csv'
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'data').exists())
real = None

if E2_MODE == 'capture':
    from quantlab.p06.cross_exchange import collect_live
    started = pd.Timestamp.now(tz='UTC')
    real_raw, real_errors = collect_live(E2_ROUNDS)
    folder = ROOT / 'data' / 'raw'; folder.mkdir(parents=True, exist_ok=True)
    real_path = folder / f"p17_quotes_{started:%Y%m%d_%H%M}.csv"
    real_raw.to_csv(real_path, index=False); real_errors.to_csv(real_path.with_name(real_path.name.replace('quotes', 'errors')), index=False)
elif E2_MODE == 'csv':
    real_path = ROOT / 'data' / 'raw' / E2_FILE
    real_raw = pd.read_csv(real_path)
    real_errors = pd.read_csv(real_path.with_name(real_path.name.replace('quotes', 'errors')))

if E2_MODE:
    real_provenance = {'source': str(real_path.relative_to(ROOT)), 'collector': 'quantlab.p06.cross_exchange.collect_live',
                       'sha256': hashlib.sha256(real_path.read_bytes()).hexdigest(), 'rows': len(real_raw), 'errors': len(real_errors),
                       'from': str(pd.to_datetime(real_raw.received_at, utc=True, format='mixed').min()),
                       'to': str(pd.to_datetime(real_raw.received_at, utc=True, format='mixed').max())}
    print(real_provenance)
    real = run_study(real_raw, qty=QTY, fee_bps=FEE_BPS, extra_bps=EXTRA_BPS, max_age_s=MAX_AGE_S)
    latency = (pd.to_datetime(real_raw.received_at, utc=True, format='mixed') - pd.to_datetime(real_raw.request_at, utc=True, format='mixed')).dt.total_seconds()
    print('Latencia de respuesta (s):', latency.groupby(real_raw.exchange).median().round(3).to_dict())
    print('Segundos:', len(real['g']), '| corte:', real['cut'], '| umbral (bps):', round(real['threshold'], 4))
    display(real['stats'].round(5)); print('R² fuera de muestra:', round(real['oos_r2'], 5))
    display(real['summary'].round(5))
    print('IC 95 % PnL medio:', real['ci'].round(6), '| sensibilidad:', real['sensitivity'].round(3).to_dict())
    print('Lead-lag (desarrollo):', lead_lag(returns_bps(real['g']).loc[lambda x: x.index < real['cut']]).idxmax().to_dict())
    print('DECISIÓN:', real['decision'])
else:
    print("E2_MODE = None: no se captura ni se lee nada. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Captura real del 20-11-2026: Kraken no anticipa a OKX</b>

**Procedencia:** `data/raw/p17_quotes_20261120.csv` (SHA-256 `8dbc1d08…65be`), capturado con `collect_live` de P06 de **10:00:47 a 10:30:47 UTC**: 1.800 rondas, 5.400 cotizaciones y **0 errores**. Latencia mediana de respuesta: Kraken 0,42 s, Bitvavo 0,44 s y OKX 0,64 s.

| | Simulación | Captura real |
|---|---|---|
| Segundos sincronizados | 7.199 | 1.800 |
| Oportunidades observadas (dev / val) | 843 / 561 | **214 / 143** (todas) |
| Lag de mayor correlación Kraken → OKX | +3 s | **0 s** (0,38) |
| IC en validación | 0,574 | **0,004** (p = 0,95) |
| R² fuera de muestra | 32,9 % | **−0,02 %** |
| Señales / ejecuciones / fallos de profundidad | 158 / 143 / 15 | 30 / **15 / 15** |
| PnL bruto / neto condicionado | +9,95 / −183,55 € | −0,11 / **−20,94 €** |
| Neto medio por ejecución | −21,3 bps | **−24,8 bps** |
| IC 95 % del PnL medio (€/oportunidad) | [−0,373; −0,299] | **[−0,234; −0,096]** |
| Criterio estadístico / económico | sí / no | **no / no** |

- **No hay retraso que aprovechar:** la correlación entre Kraken y OKX es máxima en el **mismo segundo** (0,38). Con nuestro muestreo de 1 s y 0,4–0,6 s de latencia, no se ve que Kraken vaya por delante.
- **La mitad de las señales no tiene profundidad:** el mejor ask de OKX solo tenía 0,01 BTC o más en el 70 % de las cotizaciones (E3).
- Una sola sesión de 30 minutos un jueves por la mañana: no representa otros horarios ni días.

</div>

### E3 · Auditoría por exchange

In [ ]:
print(inspect.getsource(audit_capture))

display(audit_capture(quotes, g, QTY, MAX_AGE_S).round(3))
if real is not None:
    display(audit_capture(real['quotes'], real['g'], QTY, MAX_AGE_S).round(3))
    okx = real_raw[real_raw.exchange == 'okx']
    skew = (pd.to_datetime(okx.exchange_at, utc=True, format='mixed') - pd.to_datetime(okx.received_at, utc=True, format='mixed')).dt.total_seconds()
    print(f"OKX exchange_at − received_at: mediana {skew.median():.3f} s | máximo {skew.max():.3f} s | positivo en el {(skew > 0).mean():.1%} de las filas")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Sin datos futuros, pero con un reloj desajustado</b>

**Captura real:**

| | Kraken | OKX | Bitvavo |
|---|---|---|---|
| Filas / duplicados | 1.800 / 0 | 1.800 / 0 | 1.800 / 0 |
| Hueco máximo entre recepciones | 1,16 s | 1,43 s | 1,27 s |
| Spread mediano | 0,018 bps | **2,38 bps** | 0,18 bps |
| Spread máximo | 4,24 bps | **11,06 bps** | 2,31 bps |
| Mejor bid / ask con ≥ 0,01 BTC | 78 % / 83 % | 87 % / **70 %** | 98 % / 87 % |
| Cobertura de la rejilla | 100 % | 100 % | 100 % |
| Edad media / máxima en la rejilla | 0,43 / 1,10 s | 0,61 / 1,32 s | 0,44 / 1,03 s |

- **Ninguna rejilla usa datos futuros:** todas las edades son ≥ 0 y cada `received_at` es anterior o igual a su segundo. Ningún hueco supera los 1,5 s de antigüedad máxima.
- **OKX es el mercado más caro y menos profundo:** su spread mediano es 13 veces el de Bitvavo y su mejor ask tiene menos de 0,01 BTC en el 30 % de las cotizaciones. Eso explica los fallos de profundidad.
- **El reloj local no está sincronizado con el de OKX:** en el 37 % de las filas, la hora del evento de OKX (`exchange_at`) es **posterior** a nuestra recepción, hasta 0,42 s. Es imposible, así que el reloj del equipo va retrasado. Por eso el análisis usa solo `received_at`: el orden entre exchanges con relojes distintos no es fiable a menos de un segundo.
- En la simulación, la auditoría solo encuentra lo programado: ~0,8 % de mensajes perdidos (44–57 huecos de 2 s) y spreads constantes.

</div>

### E4 · Lead-lag por sesiones y una señal OBI alternativa
Dividimos el desarrollo en dos sesiones consecutivas.

**Señal OBI propuesta antes de mirar:** el desequilibrio del mejor nivel de Kraken (`obi`) en el instante de decisión, porque si Kraken es el líder, la presión de su libro debería anticipar el movimiento. Sin parámetros que ajustar: se mide su IC con el retorno futuro de OKX **solo en desarrollo**. A modo de advertencia, se prueba también el OBI de OKX.

In [ ]:
print(inspect.getsource(session_lead_lag))

display(session_lead_lag(dev_r, 2).round({'corr_max': 3, 'corr_lag0': 3}))
for name in ['obi', 'venue_obi']:
    x = dev.dropna(subset=[name]); m = fit_hac(x, name)
    print(f"Desarrollo · {name}: IC {x[name].corr(x.target_bps):.4f} | p HAC {m.pvalues[name]:.4f} | n {len(x)}")
if real is not None:
    real_r = returns_bps(real['g']); display(session_lead_lag(real_r[real_r.index < real['cut']], 2).round({'corr_max': 3, 'corr_lag0': 3}))
    for name in ['obi', 'venue_obi']:
        x = real['dev'].dropna(subset=[name]); m = fit_hac(x, name)
        print(f"Real · desarrollo · {name}: IC {x[name].corr(x.target_bps):.4f} | p HAC {m.pvalues[name]:.4f} | n {len(x)}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · El lead-lag es estable en la simulación y nulo en la realidad</b>

**Lead-lag por sesiones:**

| Sesión de desarrollo | Simulación: OKX | Bitvavo | Real: OKX | Bitvavo |
|---|---|---|---|---|
| 1.ª | +3 s (0,98) | +2 s (0,98) | **0 s** (0,37) | **−2 s** (0,36) |
| 2.ª | +3 s (0,98) | +2 s (0,98) | **0 s** (0,41) | **−1 s** (0,50) |

- En la simulación, el retraso programado aparece igual en las dos sesiones. En los datos reales, Kraken y OKX se mueven **en el mismo segundo**, y Bitvavo va **por delante** de Kraken (lag negativo) en las dos sesiones: si alguien descubre el precio aquí, no es Kraken.

**Señal OBI:**

- **OBI de Kraken (propuesta):** IC −0,038 en la simulación (p HAC = 0,26) y −0,002 en la captura real (p = 0,98). No anticipa nada.
- **OBI de OKX (segunda prueba):** IC +0,063 (p HAC = 0,06) en la simulación, aunque los tamaños del generador son **ruido puro**. Es justo el riesgo de probar varias señales en la misma muestra: tarde o temprano una sale "casi significativa" (ver E5).

</div>

### E5 · Null por desplazamientos circulares y una muestra nueva
Desplazar el objetivo circularmente un número aleatorio de posiciones rompe la relación con la señal, pero **conserva la autocorrelación** de cada serie. Barajar observaciones individuales destruye esa dependencia: si las dos series son persistentes, el null barajado sale demasiado estrecho y la prueba da falsos positivos.

In [ ]:
print(inspect.getsource(circular_shift_null))

display(pd.DataFrame({name: circular_shift_null(dev[name], dev.target_bps, 1000, 10, SEED + 2) for name in ['signal_bps', 'obi', 'venue_obi']}).T.round(4))
rng = np.random.default_rng(0)
walk = circular_shift_null(np.cumsum(rng.normal(size=400)), np.cumsum(rng.normal(size=400)), 1000, 10, SEED + 2)
print(f"Dos paseos aleatorios independientes: corr {walk['observed']:.3f} | desv. del null circular {walk['sd_circular']:.3f} frente a barajado {walk['sd_shuffle']:.3f}")
# Muestra NUEVA (otra semilla) para la señal surgida en E4: no se usa la validación.
new_g = synchronize(validate(synthetic_quotes(seed=SEED + 1)), EXCHANGES, MAX_AGE_S)
new_op = opportunities(new_g, new_g.index[-1], LOOKBACK, DELAY, HOLD, QTY)
new = new_op[new_op.observed].dropna(subset=['venue_obi'])
print(f"Muestra nueva (semilla {SEED + 1}): OBI de OKX IC {new.venue_obi.corr(new.target_bps):.4f} | p HAC {fit_hac(new, 'venue_obi').pvalues['venue_obi']:.4f} | n {len(new)}")
if real is not None:
    display(pd.DataFrame({name: circular_shift_null(real['dev'][name], real['dev'].target_bps, 1000, 10, SEED + 2) for name in ['signal_bps', 'obi', 'venue_obi']}).T.round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Lo que no se reproduce en una muestra nueva no es una señal</b>

| Señal (desarrollo) | Simulación: corr | p circular | Real: corr | p circular |
|---|---|---|---|---|
| Retorno de Kraken en 3 s | **0,591** | **0,001** | −0,011 | 0,55 |
| OBI de Kraken | −0,038 | 0,86 | −0,002 | 0,53 |
| OBI de OKX | +0,063 | **0,04** | +0,017 | 0,45 |

- La señal principal es significativa en la simulación (ninguno de los 1.000 desplazamientos la iguala) y nula con datos reales.
- **El OBI de OKX sale con p = 0,04 en la simulación siendo ruido puro.** En una muestra nueva (semilla 1702) baja a IC 0,047 y p = 0,07, y en 20 semillas más su correlación media es 0,000. Era un falso positivo de la muestra.
- **Por qué no barajar:** dos paseos aleatorios independientes tienen aquí una correlación de 0,63. El null circular (desviación 0,42) la trata como normal; el barajado (desviación 0,05) la declararía significativa, porque "cree" que hay más información independiente de la que hay. Aquí las series son casi independientes (autocorrelaciones < 0,03 en la simulación, < 0,21 en el OBI real), así que los dos nulls coinciden.

</div>

### E6 · Salida parcial, inventario y comisiones reales
En lugar de marcar como desconocida una señal sin profundidad, `partial_fill_engine` compra lo que haya en el mejor ask, vende cada segundo lo que admita el mejor bid durante 5 s más, y deja el resto como inventario abierto valorado al último mid.

**Comisiones de OKX:** su tabla oficial de comisiones se carga con JavaScript y **no la pudimos leer**. Según las búsquedas, un usuario regular paga 0,10 % taker, y una cuenta solo spot en el EEE paga 0,20 %. Se evalúan los dos niveles, sin darlos por verificados. Las de Kraken (0,80 % taker en el primer nivel) sí se verificaron en su web en P01 y P02.

In [ ]:
print(inspect.getsource(partial_fill_engine))

for label, study in [('Simulación', {'g': g, 'val': val, 'threshold': threshold})] + ([('Real', real)] if real is not None else []):
    for fee in (10., 20.):
        pf = partial_fill_engine(study['g'], study['val'], study['threshold'], QTY, fee, EXTRA_BPS, 5)
        base = evaluate(study['val'], study['threshold'], QTY, fee, EXTRA_BPS)
        print(f"{label} · {fee:.0f} bps: {len(pf)} operaciones ({int(pf.partial_entry.sum())} con entrada parcial, {int(pf.used_wait.sum())} esperan para vender, "
              f"{int((pf.inventory_open > 1e-12).sum())} con inventario abierto) | realizado {pf.realized_eur.sum():.2f} € | "
              f"motor original: {int(base.filled.sum())} ejecuciones, {base.net_pnl.sum():.2f} € condicionados")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Ejecutar lo que falta empeora el resultado</b>

| | Comisión | Operaciones | Entrada parcial | PnL realizado | Motor original (condicionado) |
|---|---|---|---|---|---|
| Simulación | 10 bps | 155 | 8 | **−195,74 €** | −183,55 € (143 ejecuciones) |
| Simulación | 20 bps | 155 | 8 | −378,75 € | −355,56 € |
| Real | 10 bps | 30 | **14** | **−24,60 €** | −20,94 € (15 ejecuciones) |
| Real | 20 bps | 30 | 14 | −44,50 € | −37,84 € |

- Al ejecutar también las señales sin profundidad completa, la pérdida **aumenta**: el motor original no estaba ocultando operaciones ganadoras. Lo que ocultaba eran pérdidas.
- En los datos reales, **14 de 30** operaciones solo pudieron comprar parte de los 0,01 BTC. Ninguna se quedó con inventario abierto tras 5 s de espera.
- **Comparación con P08:** el coste de una ida y vuelta sin movimiento es de unos 22 bps con 10 bps por lado (P08: 21,67 bps para empatar). Con la tarifa EEE solo spot (20 bps) serían más de 40. Una señal que anticipa ~1 bp no lo cubre.
- **Supuesto:** el tamaño visible del mejor nivel se renueva cada segundo. Con snapshots L1 no sabemos si otra orden ya lo había consumido.

</div>

### E7 · Métricas de P09 y atribución de P12, sin anualizar
Se informan por oportunidad: una captura de 30 minutos o 2 horas no permite extrapolar a un año. El benchmark pertinente es **siempre long en OKX** en las mismas ventanas y con los mismos costes.

In [ ]:
print(inspect.getsource(opportunity_metrics))

def e7_table(study, pv, bench, all_op):
    n_val = int((all_op.split == 'val').sum())
    rows = {'Señal': opportunity_metrics(pv.net_bps, pv.net_pnl, INITIAL_EQUITY), 'Siempre long': opportunity_metrics(bench.net_bps, bench.net_pnl, INITIAL_EQUITY)}
    both = pd.concat([pv.net_bps.rename('s'), bench.net_bps.rename('b')], axis=1).dropna()
    alpha, beta = manual_alpha_beta(both.s, both.b)
    print(f"{study}: alpha {alpha:.2f} bps y beta {beta:.3f} frente a siempre long ({len(both)} oportunidades comunes) | "
          f"ventanas sin datos {n_val - len(pv)} de {n_val} ({(n_val - len(pv)) / n_val:.1%}) | "
          f"señales sin profundidad {int(pv.failed_depth.sum())} de {int(pv.intent.sum())} ({pv.failed_depth.sum() / pv.intent.sum():.1%})")
    return pd.DataFrame(rows).T

display(e7_table('Simulación', pv, bench, all_op).round(4))
if real is not None:
    display(e7_table('Real', real['pv'], real['bench'], real['all_op']).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Menos pérdida que siempre long, pero por estar menos expuesta</b>

| | Simulación: señal | Siempre long | Real: señal | Siempre long |
|---|---|---|---|---|
| Oportunidades con resultado | 546 | 515 | 128 | 71 |
| Media por oportunidad | −5,59 bps | −22,57 bps | −2,90 bps | −24,34 bps |
| Desviación | 9,42 bps | 1,68 bps | 8,01 bps | 1,66 bps |
| Media / desviación | −0,59 | −13,46 | −0,36 | −14,68 |
| Acierto de las operaciones | 0 % | 0 % | 0 % | 0 % |
| Caída máxima del capital | 18,4 % | 69,9 % | 2,1 % | 9,7 % |

- La señal "pierde menos" que siempre long solo porque **opera menos**: en el 72 % de las oportunidades simuladas no hay señal y el PnL es 0. Por operación, pierde lo mismo (unos 21–25 bps).
- **Atribución frente a siempre long:**
  - En la simulación, alpha de −59,6 bps y beta de −2,38, por el mismo efecto de selección de la sección 9. No es una cobertura: la señal elige las ventanas que más se mueven y paga el coste en ellas.
  - Con datos reales sale alpha +17,1 bps y beta 0,92 (71 oportunidades), pero no es informativo: el benchmark casi no varía (desviación de 1,7 bps, siempre pierde el coste), así que la regresión está mal condicionada.
- **Datos descartados:**
  - **Simulación:** 14 de 575 ventanas de validación (2,4 %) sin datos y 15 de 158 señales (9,5 %) sin profundidad.
  - **Real:** 0 de 143 ventanas sin datos, pero **15 de 30 señales (50 %)** sin profundidad.
- La media/desviación es por oportunidad, no un Sharpe anual.

</div>

### E8 · Informe de una página

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Informe: ¿anticipa Kraken a OKX en BTC/EUR?</b>

| Apartado | Contenido |
|---|---|
| **Hipótesis** | El retorno del mid de Kraken en 3 s anticipa el retorno del mid de OKX de t+1 a t+5 (H1: pendiente > 0), y una regla long-only sobre OKX gana después de costes. |
| **Fuente** | Simulación con un retraso programado de 3 s (semilla 1701) y una captura real de Kraken, OKX y Bitvavo con `collect_live` de P06 (`data/raw/p17_quotes_20261120.csv`, SHA-256 `8dbc1d08…65be`). |
| **Unidades** | Precios en EUR/BTC, cantidades en BTC, retornos y costes en bps, PnL en EUR por 0,01 BTC. Tiempos en UTC según la **recepción**. |
| **Periodo** | Simulación: 2 h (7.199 s). Real: 20-11-2026, de 10:00:47 a 10:30:47 UTC (1.800 s). Corte 60 % / 40 %. |
| **Método** | Rejilla causal de 1 s (edad máxima 1,5 s), decisiones cada 5 s sin solapamiento, umbral p70 de desarrollo, regresión HAC, R² fuera de muestra con modelo congelado, ejecución al ask/bid con profundidad L1, bootstrap por bloques y null circular. |
| **Evidencia** | Simulación: R² fuera de muestra del 32,9 %. Real: lag máximo en 0 s, IC de validación 0,004 (p = 0,95), R² fuera de muestra −0,02 %. |
| **Costes** | 10 bps por lado + 1 bp de reserva adversa: unos 22 bps por ida y vuelta, más el spread de OKX (2,4 bps de mediana en la captura). Con 20 bps (tarifa EEE solo spot, no verificada en fuente primaria), el doble. |
| **Resultado** | Todas las operaciones pierden en los dos casos: −21,3 bps (simulación) y −24,8 bps (real) por ejecución. IC del PnL medio por debajo de 0. |
| **Límites** | Una sola sesión real de 30 min; snapshots REST de 1 s con 0,4–0,6 s de latencia; reloj local desajustado respecto a OKX; profundidad solo del mejor nivel; sin colas, sin latencia de órdenes y sin comisiones verificadas de OKX. |
| **Decisión** | **No aprobar.** Simulación: evidencia estadística sí, económica no. Real: ni una ni otra. |
| **Próximos experimentos** | Captura por WebSocket con secuencias y reloj NTP; varias sesiones y horarios; evaluar el protocolo E1 en una captura nueva; tarifas maker y su riesgo de no ejecución. |

Un resultado negativo reproducible es una investigación válida: sabemos **por qué** no funciona (no hay retraso real que aprovechar, y aun con el retraso simulado la ida y vuelta cuesta unas 19 veces el movimiento bruto medio que anticipa la señal).

</div>

## 12. Cierre del módulo

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P17</b>

**¿Anticipa Kraken a OKX?** En la simulación, sí, porque se programa un retraso de 3 s (R² fuera de muestra del 32,9 %). En la captura real del 20-11-2026, no: los dos mercados se mueven en el mismo segundo y es Bitvavo quien va por delante de Kraken.

**¿Es viable una regla long-only sobre OKX?** No, ni siquiera en la simulación favorable. Todas las operaciones pierden: la ida y vuelta cuesta unos 22 bps y la señal anticipa ~1 bp. Ejecutar parcialmente las señales sin profundidad aumenta la pérdida.

**¿Qué aprendimos del proceso?** El protocolo fijado de antemano impidió elegir el lag con la validación. La auditoría encontró que el reloj local no está sincronizado con OKX. Y el null circular y la muestra nueva desmontaron una señal OBI que parecía significativa.

El cuaderno integra P02–P06 (cotizaciones, indicadores, retornos y desfases), P08–P09 (costes y riesgo), P11–P13 (validación, atribución y microestructura). P07 y P10/P14–P16 aportan extensiones para derivados, macroeconomía y factores; no son imprescindibles para una hipótesis intradía spot y no se fuerzan artificialmente en ella. El examen teórico y práctico sigue pendiente. La ejecución sintética no acredita una oportunidad real.

📄 Resumen en `docs/conclusiones/P17_integrated_research.md`

</div>

**Reproducción:** ejecutar todas las celdas en Colab o Jupyter con numpy, pandas, matplotlib, statsmodels. Por defecto no se consulta ningún endpoint. Para el E2, `E2_MODE='capture'` captura unos 30 minutos con P06, o `'csv'` reutiliza la captura guardada en `data/raw/` (no versionada).